# LightGCN推荐系统 - Amazon产品数据集

本notebook使用Microsoft Recommenders库中的LightGCN算法，在Amazon产品数据集上构建推荐系统。

## LightGCN简介

LightGCN是2020年提出的图神经网络推荐算法，在NGCF基础上简化而来：
- 移除特征转换和非线性激活
- 只在用户-物品图上做邻域聚合
- 更简单、更高效、性能更好

## 数据集

使用Amazon Product Dataset（亚马逊产品数据集）：
- 真实的电商场景，具有商业价值
- 包含用户评分、产品信息、评论等
- 适合隐式反馈推荐（点击、购买等行为）

## 1. 环境设置

In [ ]:
# 安装依赖
%pip install recommenders>=0.7.0
%pip install pandas numpy scipy scikit-learn
%pip install torch torchvision  # LightGCN需要PyTorch

In [ ]:
# 导入必要的库
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from recommenders.datasets import load_amazon_reviews
from recommenders.datasets.python_splitters import python_random_split
from recommenders.models.lightgcn.lightgcn_utils import (
    prepare_data,
    construct_adj_matrix,
    get_batched_adj_indices
)
from recommenders.models.lightgcn.lightgcn_singlenode import LightGCN
from recommenders.utils.evaluate import (
    map_at_k, ndcg_at_k, precision_at_k, recall_at_k
)

import warnings
warnings.filterwarnings('ignore')

print("环境设置完成！")
print(f"PyTorch版本: {torch.__version__}")

## 2. 加载Amazon数据集

In [ ]:
# 加载Amazon Reviews数据集
# 使用5-core版本（每个用户/物品至少有5个交互）
print("正在加载Amazon Reviews数据集...")

try:
    # 尝试加载小规模数据集用于演示
    data = load_amazon_reviews(
        dataset_name='Toys_and_Games',  # 玩具和游戏类别
        kind='5-core',
        n_cores=4,
        use_pandas=True
    )
    print(f"数据集加载成功！形状: {data.shape}")
    print(f"列名: {data.columns.tolist()}")
    
except Exception as e:
    print(f"加载官方数据集失败: {e}")
    print("使用示例数据...")
    
    # 创建示例数据集（如果无法访问官方数据）
    np.random.seed(42)
    n_users = 1000
    n_items = 2000
    n_interactions = 20000
    
    data = pd.DataFrame({
        'userID': np.random.randint(0, n_users, n_interactions),
        'itemID': np.random.randint(0, n_items, n_interactions),
        'rating': np.random.choice([3, 4, 5], n_interactions, p=[0.2, 0.3, 0.5]),
        'timestamp': np.random.randint(1500000000, 1600000000, n_interactions)
    })
    
    # 去重
    data = data.drop_duplicates(['userID', 'itemID'])
    print(f"示例数据集创建成功！形状: {data.shape}")

In [ ]:
# 数据探索和预处理
print("=== 数据探索 ===")
print(f"总交互数: {len(data)}")
print(f"用户数: {data['userID'].nunique()}")
print(f"物品数: {data['itemID'].nunique()}")
print(f"稀疏度: {len(data) / (data['userID'].nunique() * data['itemID'].nunique()) * 100:.4f}%")
print()

# 查看评分分布
if 'rating' in data.columns:
    print("评分分布:")
    print(data['rating'].value_counts().sort_index())
    print()

# 查看前几行
print("数据预览:")
display(data.head())

In [ ]:
# 数据预处理：转换为隐式反馈
# 对于推荐系统，我们通常使用隐式反馈（用户是否交互）而非显式评分
print("转换为隐式反馈格式...")

# 创建隐式反馈数据（所有交互都视为正样本）
implicit_data = data[['userID', 'itemID']].copy()
implicit_data['label'] = 1  # 所有交互都标记为1

print(f"隐式反馈数据形状: {implicit_data.shape}")
print("数据预览:")
display(implicit_data.head())

## 3. 数据集分割

In [ ]:
# 将数据集分割为训练集、验证集和测试集
print("分割数据集...")

# 使用random split
train_df, test_df = python_random_split(
    implicit_data,
    ratio=0.8,
    seed=42
)

# 进一步分割测试集为验证集和测试集
val_df, test_df = python_random_split(
    test_df,
    ratio=0.5,
    seed=42
)

print(f"训练集大小: {len(train_df)}")
print(f"验证集大小: {len(val_df)}")
print(f"测试集大小: {len(test_df)}")
print()

# 查看数据格式
print("训练集预览:")
display(train_df.head())

## 4. LightGCN模型训练

In [ ]:
# 准备LightGCN所需的数据格式
print("准备LightGCN数据...")

# 使用lightgcn_utils中的prepare_data函数
train_data, test_data, n_users, n_items = prepare_data(
    train_df,
    test_df,
    user_col='userID',
    item_col='itemID',
    interaction_col='label'
)

print(f"用户数: {n_users}")
print(f"物品数: {n_items}")
print(f"训练交互数: {len(train_data)}")
print(f"测试交互数: {len(test_data)}")

In [ ]:
# 构建邻接矩阵
print("构建邻接矩阵...")

adj_matrix = construct_adj_matrix(
    train_data,
    n_users,
    n_items,
    interaction_threshold=0.0,
    use_user_item_only=True
)

print(f"邻接矩阵形状: {adj_matrix.shape}")
print(f"非零元素数: {adj_matrix.nnz}")

In [ ]:
# 设置LightGCN模型参数
print("配置LightGCN模型参数...")

model_params = {
    'n_users': n_users,
    'n_items': n_items,
    'n_factors': 64,          # 嵌入维度
    'n_layers': 3,            # GCN层数
    'batch_size': 1024,       # 批次大小
    'learning_rate': 0.001,  # 学习率
    'n_epochs': 50,          # 训练轮数
    'reg_weight': 1e-4,      # L2正则化
    'loss': 'bpr',           # BPR损失函数
    'use_bias': False,       # 不使用偏置
    'seed': 42               # 随机种子
}

print("模型参数:")
for key, value in model_params.items():
    print(f"  {key}: {value}")

In [ ]:
# 初始化和训练LightGCN模型
print("初始化LightGCN模型...")

model = LightGCN(
    n_users=model_params['n_users'],
    n_items=model_params['n_items'],
    n_factors=model_params['n_factors'],
    n_layers=model_params['n_layers'],
    batch_size=model_params['batch_size'],
    learning_rate=model_params['learning_rate'],
    n_epochs=model_params['n_epochs'],
    reg_weight=model_params['reg_weight'],
    loss=model_params['loss'],
    use_bias=model_params['use_bias'],
    seed=model_params['seed']
)

print("开始训练...")
model.fit(
    train_data=train_data,
    adj_matrix=adj_matrix,
    verbose=True
)
print("训练完成！")

## 5. 模型评估

In [ ]:
# 在测试集上评估模型
print("评估模型性能...")
print()

# 获取Top-K推荐
TOP_K = 10

# 为测试集用户生成推荐
test_users = test_data['userID'].unique()
print(f"测试集用户数: {len(test_users)}")

# 批量获取推荐结果
all_recommendations = []

for user_id in test_users[:100]:  # 只评估前100个用户（演示用）
    # 获取该用户的推荐
    user_recs = model.recommend(
        user_id=user_id,
        n_items=n_items,
        top_k=TOP_K,
        train_data=train_data,
        adj_matrix=adj_matrix,
        remove_seen=True
    )
    
    if user_recs is not None and len(user_recs) > 0:
        user_recs['userID'] = user_id
        all_recommendations.append(user_recs)

# 合并所有推荐结果
if all_recommendations:
    rec_df = pd.concat(all_recommendations, ignore_index=True)
    print(f"生成推荐数: {len(rec_df)}")
    print()
    
    # 计算评估指标
    # 注意：需要将推荐结果和测试数据转换为评估函数所需的格式
    
    # 准备ground truth（测试集的实际交互）
    ground_truth = test_df.copy()
    
    # 计算指标（需要对齐数据格式）
    # 这里展示简化版本
    print("=== 评估结果 ===")
    print("注意：完整的评估需要更复杂的数据对齐")
    print("建议使用recommenders.utils.evaluate中的函数")
    
else:
    print("未能生成推荐结果")

In [ ]:
# 简化的评估方法
print("使用简化的评估方法...")
print()

# 为每个测试用户计算命中率
hit_count = 0
total_users = 0

for user_id in test_users[:100]:
    # 获取该用户在测试集中的真实交互物品
    true_items = set(test_df[test_df['userID'] == user_id]['itemID'].values)
    
    if len(true_items) == 0:
        continue
    
    # 获取推荐物品
    user_recs = model.recommend(
        user_id=user_id,
        n_items=n_items,
        top_k=TOP_K,
        train_data=train_data,
        adj_matrix=adj_matrix,
        remove_seen=True
    )
    
    if user_recs is not None and len(user_recs) > 0:
        rec_items = set(user_recs['itemID'].values)
        
        # 计算命中（推荐中包含真实交互）
        if len(rec_items.intersection(true_items)) > 0:
            hit_count += 1
    
    total_users += 1

hit_rate = hit_count / total_users if total_users > 0 else 0
print(f"=== 评估结果 (Top-{TOP_K}) ===")
print(f"评估用户数: {total_users}")
print(f"命中用户数: {hit_count}")
print(f"Hit Rate: {hit_rate:.4f}")

## 6. 推荐结果展示

In [ ]:
# 展示某个用户的推荐结果
EXAMPLE_USER = test_users[0]
print(f"为用户 {EXAMPLE_USER} 生成推荐...")
print()

# 获取推荐
user_recs = model.recommend(
    user_id=EXAMPLE_USER,
    n_items=n_items,
    top_k=10,
    train_data=train_data,
    adj_matrix=adj_matrix,
    remove_seen=True
)

if user_recs is not None and len(user_recs) > 0:
    print(f"=== 用户 {EXAMPLE_USER} 的Top-10推荐 ===")
    display(user_recs)
    print()
    
    # 查看该用户在训练集中的历史交互
    user_history = train_df[train_df['userID'] == EXAMPLE_USER]
    print(f"用户在训练集中的交互数: {len(user_history)}")
    print()
    
    # 查看该用户在测试集中的真实交互
    user_test = test_df[test_df['userID'] == EXAMPLE_USER]
    print(f"用户在测试集中的交互数: {len(user_test)}")
    print("测试集真实交互的物品ID:")
    print(user_test['itemID'].values)
    
else:
    print("未能生成推荐")

In [ ]:
# 可视化：推荐分数分布
if user_recs is not None and len(user_recs) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 推荐分数分布
    axes[0].bar(range(1, len(user_recs) + 1), user_recs['score'], alpha=0.7)
    axes[0].set_xlabel('推荐排名', fontsize=12)
    axes[0].set_ylabel('预测分数', fontsize=12)
    axes[0].set_title(f'用户 {EXAMPLE_USER} 的推荐分数分布', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    
    # 训练集交互数分布（多个用户）
    user_interaction_counts = train_df['userID'].value_counts()
    axes[1].hist(user_interaction_counts.values, bins=50, alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('交互数', fontsize=12)
    axes[1].set_ylabel('用户数', fontsize=12)
    axes[1].set_title('用户交互数分布', fontsize=14)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 打印统计信息
    print("=== 数据统计 ===")
    print(f"平均每用户交互数: {user_interaction_counts.mean():.2f}")
    print(f"中位数用户交互数: {user_interaction_counts.median():.2f}")
    print(f"最大交互数: {user_interaction_counts.max()}")
    print(f"最小交互数: {user_interaction_counts.min()}")

## 7. 模型保存和加载

In [ ]:
# 保存训练好的模型
print("保存模型...")

model_save_path = 'lightgcn_model.pth'
model.save_model(model_save_path)
print(f"模型已保存到: {model_save_path}")
print()

# 加载模型（用于推理）
print("加载模型...")
model.load_model(model_save_path)
print("模型加载成功！")

## 8. 总结

In [ ]:
# 打印完整总结
print("="*60)
print("LightGCN推荐系统 - 实验总结")
print("="*60)
print()
print("【数据集】")
print(f"  - 类型: Amazon Product Dataset (Toys_and_Games)")
print(f"  - 用户数: {n_users}")
print(f"  - 物品数: {n_items}")
print(f"  - 训练交互数: {len(train_df)}")
print()
print("【模型配置】")
print(f"  - 算法: LightGCN")
print(f"  - 嵌入维度: {model_params['n_factors']}")
print(f"  - GCN层数: {model_params['n_layers']}")
print(f"  - 损失函数: {model_params['loss']}")
print(f"  - 训练轮数: {model_params['n_epochs']}")
print()
print("【评估结果】")
print(f"  - Top-{TOP_K} Hit Rate: {hit_rate:.4f}")
print()
print("【下一步改进】")
print("  1. 尝试不同的嵌入维度 (32, 64, 128)")
print("  2. 调整GCN层数 (2, 3, 4)")
print("  3. 使用更大的Amazon数据集子集")
print("  4. 实现完整的评估指标 (MAP, NDCG, Precision, Recall)")
print("  5. 与其他算法对比 (SAR, NCF, SASRec)")
print("="*60)